# Data Cleansing v2

update: 20250616

Data Cleansing involves cleaning Lv2 data and performing imputation of missing values to obtain an organized dataset that can be directly used for model training.

Steps:
- [common step 0] Database Overview
- [common step 1] Quality Control (features and samples)
- [common step 2] Imputation of Missing Values
- [common step 3] Age Linear Transformation

The final imputed dataset obtained: imputed_training_set.csv

## Glossary

- Feature: CpG site, cg probe, probe, feature
- Sample: sample, case
- GSEmtx: A data format defined in this study, adapted from the GEOmatrix file. Each row represents a sample, and each column represents annotation information and features, with annotation information starting with an exclamation mark.

In [ ]:

# set environment, necessary:

#PATH_database = Enter the path '/Data-Lv_2_formatted/'
#PATH_output = Enter the save path


In [ ]:
import math
import pandas as pd
import numpy as np
import os
import pickle
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer

In [ ]:
file_list = os.listdir(PATH_database)

In [ ]:
# Get the non-comment column names (i.e., probe names) in the dataset.
# df specifically refers to a pandas DataFrame object in this document.
# Note: Columns in df that start with an exclamation mark are comment columns, while those that do not are probe columns.
# Make sure that comment columns are marked with an exclamation mark before calling this method.
def probe_list(df):
    return df.columns[~df.columns.str.contains('!')]

# Filter out probe columns from the dataset df that contain fewer missing values than the threshold (default is 10) and return a list of column names.
# df is a DNAm dataset (where annotation columns are marked with an exclamation mark !); threshold is the missing value threshold, and probes with a number of missing values exceeding this threshold will be removed; report controls on-screen feedback.
def good_probes(df,threshold=10,report=True):
    probe_null_counts = df[probe_list(df)].isnull().sum(axis=0)
    reserved_probes = probe_null_counts[probe_null_counts.values<=threshold].index
    if report == True:
        print('Before\trestriction:',len(probe_list(df)),'probes')
        print('After\trestriction:',len(reserved_probes),'probes')
    return reserved_probes

# Extract GSE accession from strings containing GSE accession (such as filenames)
# Input the string to be extracted, return the extracted string or None.
def gse_acc_from_filename(file_name,patrn=r'(GSE\d+)\D'):
    pattern = re.compile(patrn,re.I)
    extracted_gse_acc = pattern.findall(file_name)
    if len(extracted_gse_acc) == 1:
        return extracted_gse_acc[0]
    else:
        print('error!: multiple gse acc were detected!')
        return None
    
# Exam
def database_abstract(PATH,files=None,good_prb_threshold=10):
    if files == None:
        files = os.listdir(PATH)
    abstract = []
    age = []
    i = 0
    for file in files:
        i += 1
        print('Processing',str(i)+'/'+str(len(files)),'\t'+file,'...')
        df = pd.read_pickle(PATH+file)
        acc = gse_acc_from_filename(file)
        datatype = ','.join(tuple(map(lambda x: str(x),list(set(df[probe_list(df)].dtypes)))))
        sample_counts = len(df.index)
        probe_counts = len(probe_list(df))
        good_prb = good_probes(df,threshold=good_prb_threshold,report=False)
        good_prb_counts = len(good_prb)
        abstract.append({'file_name':file,'gse_acc':acc,'datatype':datatype,'sample_counts':sample_counts,'probe_counts':probe_counts,'good_prb_counts':good_prb_counts})
        age.append(list(df['!age']))
    return pd.DataFrame(abstract),pd.DataFrame(age,index=files)

# Using the variable cordon method, iterate through all datasets in the database to obtain the intersection of good probes across datasets. 20230510
# Returns a tuple: (probe set (set), dataset table (tuple), total number of samples (int))
# The variable cordon method can obtain as many series as possible while staying as close as possible to the lower limit of the probe set.
# brake_level controls the second cordon (the first being the variable cordon). The second cordon is calculated as the difference between the platform probe number (i.e., maximum number of probes, upper limit) and the lower limit of probes divided by brake_level. 
#The second cordon, together with the first (variable) cordon, jointly controls the rate at which probes are reduced. 
#The number of probes reduced at each step will always be within the minimum of the first and second cordons.
def common_good_probes_v3(path_database='./',null_thredshold=10,min_platform='27k',low_probe_limit=20000,brake_level=3,first_data=None,safe_data=[]): #,col_rename_dict={}):
    platforms = {'27k':27000,'450k':450000,'epic':850000}
    if min_platform not in ('27k','450k','epic'):
        print("Error in param.: 'min_platform', which should be '27k','450k' or 'epic'.")
        return
    data_list = os.listdir(path_database)
    second_cordon = (platforms[min_platform]-low_probe_limit)/brake_level
    selected_series_list = []
    sample_amount = 0
    
    if first_data != None:
        if first_data not in data_list:
            print("Error in param.: 'first_data' is not in the database.\nNote: first_data should contain Filename Extension, such as '.pkl'.")
            return
        first_data_index = data_list.index(first_data)
        data_list[0],data_list[first_data_index] = data_list[first_data_index],data_list[0]
    
    for i in range(len(data_list)):
        print('Processing ',i+1,'/',len(data_list),': ',data_list[i],' ...',sep='')
        df = pd.read_pickle(path_database+data_list[i])
        #df.rename(columns=col_rename_dict,inplace=True)
        if i == 0:
            common_probes = set(good_probes(df,threshold=null_thredshold,report=False))
            selected_series_list.append(data_list[i])
        else:
            cordon = (len(common_probes)-low_probe_limit)/(len(data_list)-i)
            temp = set(good_probes(df,threshold=null_thredshold,report=False)) & common_probes
            if len(common_probes)-len(temp) > min(cordon,second_cordon) and data_list[i] not in safe_data:
                print('[Notice]\tHigh reduction of probe ammounts, skip.')
                continue
            common_probes = temp
            selected_series_list.append(data_list[i])
        sample_amount = sample_amount + df.shape[0]
        print('[Counts]\tCurrent Probes:',len(common_probes),'\tCurrent Samples:',sample_amount)
    
    print(len(selected_series_list),'series,',len(common_probes),'probes,',sample_amount,'samples.')
    return (common_probes,tuple(selected_series_list),sample_amount)

# Convert the return value of common_good_probes into a single concatenated dataframe
def concated_common_good_probes(common_good_probes_output):#,col_rename_dict={}):
    for i in range(len(common_good_probes_output[1])):
        print('Concating ',i+1,'/',len(common_good_probes_output[1]),':',common_good_probes_output[1][i])
        df = pd.read_pickle(path_database+common_good_probes_output[1][i])
        #df.rename(columns=col_rename_dict,inplace=True)
        df_annot = df.columns[df.columns.str.contains('!')]
        df = df[list(df_annot)+list(common_good_probes_output[0])]
        if i == 0:
            combined_df = df
        else:
            combined_df = pd.concat([combined_df,df])
    return combined_df

# Data value discretization method. Input a value x, then compare it with the boundaries in quantiles=(a,b,c,...). If it is less than a, it is recorded as 0; if it is less than b but greater than a, it is recorded as 1; if it is less than c but greater than b, it is recorded as 2; and so on.
# The boundary values in the method tuple must be arranged in ascending order!
def discret(x,quantiles=(0.25,0.75)):
    for i in range(len(quantiles)):
        if x <= quantiles[i]:
            return i
    return len(quantiles)

In [ ]:
# horvath age transformation

def trafo(x,adult_age=20):
    x = (x+1)/(1+adult_age)
    if x <= 1:
        y = math.log(x)
    else:
        y = x-1
    return y

def anti_trafo(x,adult_age=20):
    if x < 0:
        return (1+adult_age)*math.exp(x)-1
    else:
        return (1+adult_age)*x+adult_age

## Common Step 0. Database Overview

In [ ]:
database_abstr,database_age = database_abstract(PATH_database,files=file_list)

In [ ]:
database_abstr.sort_values(by='good_prb_counts',ascending=False).iloc[:60]

In [ ]:
database_abstr['sample_counts'].sum()

## Common Step 1. Quality Control

In [ ]:
# Specify any Lv2 dataset as the starting point
FIRST_dataset = 'GSE147740_GPL21145_NORMAL_lv2.pkl'

# Data set designated as necessary to be retained by humans
SAFE_dataset = ['GSE147740_GPL21145_NORMAL_lv2.pkl',
                'GSE196696_GPL21145_NORMAL_lv2.pkl',
                'GSE152026_GPL21145_NORMAL_lv2.pkl',
                'GSE207927_GPL21145_NORMAL_lv2.pkl']

In [ ]:
common_gp = common_good_probes_v3(path_database=PATH_database,null_thredshold=10,min_platform='epic',low_probe_limit=700000,brake_level=1,first_data=FIRST_dataset,safe_data=SAFE_dataset)


In [ ]:
training_set = concated_common_good_probes(common_gp)
training_set

In [ ]:
# View the age distribution of the training set
plt.figure(figsize=(3,3),dpi=300)
sns.violinplot(data=np.array(training_set['!age']),inner='stick')
plt.ylabel("age")
#plt.xticks(ticks=np.arange(1),labels='Training set')

In [ ]:
training_set.to_pickle(PATH_output+'training_set.pkl')

In [ ]:
with open(PATH_output+'common_gp.pkl','wb') as f:
    pickle.dump(common_gp,f)

## Common Step 2. Missing Values Imputation (KNN)

Note: Ensure the behavior sample is listed as a feature!

In [ ]:
# Extract the sample annotation information into a new table for merging with the imputation results.
annot_df = training_set[training_set.columns[training_set.columns.str.contains('!')]]
print(annot_df)
# Extract the beta values of the samples into a new table for imputation.
dnam_df = training_set[training_set.columns[~training_set.columns.str.contains('!')]]
print(dnam_df)

In [ ]:
# KNN imputation, returning array-structured data
imputer = KNNImputer()
imputed_dnam_array = imputer.fit_transform(dnam_df)

In [ ]:
# 9h40min finished

# Convert the array returned by imputation into a DataFrame
imputed_set = pd.DataFrame(imputed_dnam_array,index=dnam_df.index,columns=dnam_df.columns)
# Recombine sample annotation information with the imputed beta values
imputed_set = pd.concat([annot_df,imputed_set],axis=1)

# autosave. remember to edit the output file name below!
imputed_set.to_pickle(PATH_output+'imputed_training_set.pkl')

imputed_set

In [ ]:
# Changes in Feature Missing Values Before and After Statistical Imputation
print('null counts before imputing:',training_set[probe_list(training_set)].isnull().sum().sum())
print('null counts after imputing:',imputed_set[probe_list(imputed_set)].isnull().sum().sum())

## Common Step 3. Age Linear Transformation

In [ ]:
imputed_set['!age'] = imputed_set['!age'].astype('float64')

In [ ]:
imputed_set['!age'].dtypes

In [ ]:
imputed_set.insert(1,'!trans_age',imputed_set['!age'].map(trafo))
imputed_set